# Batch-Level Visualization Demo

This notebook demonstrates how to visualize batch-level state variables in AnimaKinetica.

## Features:
- Plot biomass and metabolite concentrations over time
- Plot volume progression and feed/dilution rates
- Test with different operating modes (Batch, Fedbatch, Perfusion)
- Compare results across different batches

---

In [1]:
# Import necessary modules
import numpy as np
from animakinetica.core.parameters import SimulationParameters
from animakinetica.core.simulation import CellCultureSimulation
from animakinetica.models.kinetics import Kinetics
from animakinetica.models.odesystems import OperatingMode
from animakinetica.analysis.batch_visualization import (
    plot_batch_biomass_metabolites,
    plot_batch_volume_feed,
    plot_batch_pH_control
)
from animakinetica.models.feed_strategies import FeedStrategy
from animakinetica.models.perfusion_strategies import PerfusionStrategy

## 1. Initialize Simulation

Load parameters and create a simulation instance with default settings (Fedbatch mode).

In [2]:
# Load default parameters
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "fed_batch"])

# Create simulation with EXCESS kinetics and FEDBATCH mode
sim = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    feed_strategy=FeedStrategy.EXPONENTIAL,
    operating_mode=OperatingMode.FEDBATCH
)

print(f"Simulation Type: {sim.data.simulation_type}")
print(f"Kinetics Model: {sim.kinetic_model.__class__.__name__}")
print(f"Operating Mode: {sim.operating_mode}")

Loading parameters from 5 file(s) sequentially...
  [1/5] Base: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/5] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\operating_modes\fed_batch.yaml
Successfully loaded parameters from 5 file(s).
Simulation Type: 
Kinetics Model: Stoichiome

## 2. Run Single Batch Operation

Execute a single batch to test batch-level plotting functions.

In [3]:
# Run single batch
results = sim.run_simulation(number_of_batches=1)
batch1 = sim.data.batches[0]

print(f"\nBatch completed in {batch1.t_end:.1f} hours")
print(f"Final cell density: {batch1.Xv_final/1e9:.2f} x 10^6 cells/mL")
print(f"Final volume: {batch1.V_final:.2f} L")

run_simulation execution time: 46.702 ms

Batch completed in 144.0 hours
Final cell density: 9.82 x 10^6 cells/mL
Final volume: 2000.00 L


## 3. Plot Biomass and Metabolites

Visualize cell density and metabolite concentrations (glucose, glutamine, lactate, ammonia).

In [4]:
# Plot biomass and metabolites for batch 1
fig_biomass = plot_batch_biomass_metabolites(sim, batch_number=1)
fig_biomass.show()

# Alternative: using the data object method
# fig_biomass = sim.data.batches[0].plot_biomass_metabolites()
# fig_biomass.show()

### Interpretation:

**Left Panel (Cell Density):**
- Shows exponential growth followed by stationary phase
- Growth slows as nutrients deplete or inhibitors accumulate

**Right Panel (Metabolites):**
- **Glucose & Glutamine** (primary y-axis): Decrease as cells consume them
- **Lactate** (primary y-axis): Increases as waste product from glucose metabolism
- **Ammonia** (secondary y-axis): Increases from glutamine degradation

The dual y-axis allows ammonia (lower concentration) to be visible alongside other metabolites.

## 4. Plot pH control

Test the pH control visualization function.

### 4.1. Plot pH control parameters from BatchData

In [5]:
# Test with BatchData directly
fig_pH = plot_batch_pH_control(sim, batch_number=1)
fig_pH.show()

**Interpretation:**
- **pH**: Shows pH evolution throughout the batch (calculated from H+ concentration)
- **Base Addition Rate (D_base)**: The rate at which NaOH is added for pH control (1/h)
- **Dissolved CO₂**: CO2 accumulation from cellular respiration
- **Na⁺ Concentration**: Increases as NaOH base is added for pH control, contributes to osmolality

This 4-panel view gives a complete picture of pH control dynamics in the bioreactor.

### 4.2. Test with custom layout

In [6]:
# Test with custom layout
custom_layout = {'height': 500, 'width': 2000}
fig_pH_custom = plot_batch_pH_control(sim, batch_number=1, layout_kwargs=custom_layout)
fig_pH_custom.show()

## 5. Plot Volume and Feed Rate

Visualize how reactor volume and feed/dilution rate change over time.

In [7]:
# Plot volume and feed rate for batch 1
fig_volume = plot_batch_volume_feed(sim, batch_number=1)
fig_volume.show()

# Alternative: using the data object method
# fig_volume = sim.data.batches[0].plot_volume_feed()
# fig_volume.show()

### Interpretation:

**Volume (primary y-axis):**
- In Fedbatch: Volume increases as feed medium is added
- In Batch: Volume stays constant
- In Perfusion: Volume stays constant (feed = bleed)

**Dilution Rate (secondary y-axis):**
- D = F/V where F is feed rate, V is volume
- Shows the instantaneous dilution effect
- In exponential feeding: D stays constant
- In adaptive feeding: D adjusts based on cell density

## 6. Compare Multiple Batches

Run multiple batches and compare the profiles.

In [8]:
# Run 3 batches
results_multi = sim.run_simulation(number_of_batches=3)

print(f"\nCompleted {len(sim.data.batches)} batches")
for i, batch in enumerate(sim.data.batches, 1):
    print(f"Batch {i}: {batch.t_end:.1f} h, Final Xv: {batch.Xv_final/1e9:.2f} x 10^6 cells/mL")

run_simulation execution time: 117.010 ms

Completed 3 batches
Batch 1: 144.0 h, Final Xv: 9.82 x 10^6 cells/mL
Batch 2: 107.1 h, Final Xv: 9.86 x 10^6 cells/mL
Batch 3: 107.0 h, Final Xv: 9.83 x 10^6 cells/mL


In [9]:
# Plot batch 1
fig_b1 = plot_batch_biomass_metabolites(sim, batch_number=1)
fig_b1.update_layout(title_text="Batch 1: Biomass and Metabolite Progression")
fig_b1.show()

In [10]:
# Plot batch 2
fig_b2 = plot_batch_biomass_metabolites(sim, batch_number=2)
fig_b2.update_layout(title_text="Batch 2: Biomass and Metabolite Progression")
fig_b2.show()

In [11]:
# Plot batch 3
fig_b3 = plot_batch_biomass_metabolites(sim, batch_number=3)
fig_b3.update_layout(title_text="Batch 3: Biomass and Metabolite Progression")
fig_b3.show()

## 7. Test with Different Operating Modes

Compare visualization across Batch, Fedbatch, and Perfusion modes.

### 7.1. Batch Mode (No Feed)

In [12]:
# Create batch mode simulation
sim_batch = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    feed_strategy=None,
    perfusion_strategy=None,
    operating_mode=OperatingMode.BATCH
)

results_batch = sim_batch.run_simulation(number_of_batches=1)
batch_data = sim_batch.data.batches[0]
print(f"Batch mode completed in {batch_data.t_end:.1f} hours")

# Plot
fig_batch = plot_batch_biomass_metabolites(sim_batch, batch_number=1)
fig_batch.show()

fig_vol_batch = plot_batch_volume_feed(sim_batch, batch_number=1)
fig_vol_batch.show()

run_simulation execution time: 24.744 ms
Batch mode completed in 97.0 hours


### 7.2. Perfusion Mode (Continuous Feed & Bleed)

In [13]:
# Create perfusion mode simulation
params = SimulationParameters.load_parameters(["Cell_GFI", "anab_GFI", "cata_GFI", "excess_kinetics_batch", "perfusion"])
#params.set_parameter_value('process_schedule.t_max', 230)  # shorten simulation time for testing

sim_perfusion = CellCultureSimulation(
    parameters=params,
    kinetics=Kinetics.STOICHIOMETRIC,
    feed_strategy=None,
    perfusion_strategy=PerfusionStrategy.CSPR,
    operating_mode=OperatingMode.PERFUSION
)

results_perfusion = sim_perfusion.run_simulation()
batch_data = sim_perfusion.data.batches[0]
print(f"Perfusion mode completed in {batch_data.t_end:.1f} hours")

# Plot
fig_perfusion = plot_batch_biomass_metabolites(sim_perfusion, batch_number=1)
fig_perfusion.show()

fig_vol_perfusion = plot_batch_volume_feed(sim_perfusion, batch_number=1)
fig_vol_perfusion.show()

fig_pH_perfusion = plot_batch_pH_control(sim_perfusion, batch_number=1)
fig_pH_perfusion.show()

fig_gas_transfer = batch_data.plot_gas_transfer()
fig_gas_transfer.show()

Loading parameters from 6 file(s) sequentially...
  [1/6] Base: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\base_parameters.yaml
  [2/6] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\cell_properties\Cell_GFI.yaml
  [3/6] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\anab_GFI.yaml
  [4/6] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\stoichiometry\cata_GFI.yaml
  [5/6] Updating with: C:\Users\btumulero\OneDrive - Delft University of Technology\2. Code\animakinetica\src\animakinetica\config\parameter_sets\excess_kinetics\excess_kinetics_batch.yaml
  [6/6] Updating with: C:\Users\btumulero\OneDrive - Delft University of Techno

## 8. Custom Layout Options

Demonstrate customization of plot appearance.

In [14]:
# Custom layout for publication-ready figures
custom_layout = {
    'height': 500,
    'width': 1600,
    'font': dict(size=14, family='Arial'),
    'title_font': dict(size=16),
    'template': 'simple_white'
}

fig_custom = plot_batch_biomass_metabolites(
    sim, 
    batch_number=1,
    layout_kwargs=custom_layout
)
fig_custom.show()